[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/06_%E6%99%82%E7%B3%BB%E5%88%97%E3%81%A8%E5%A0%B4%E5%90%88%E3%81%AE%E6%95%B0.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計4級-06. 時系列データと場合の数（樹形図）

**できるようになること**
- 増減率・指数・移動平均で時系列を読める
- 場合の数を樹形図/列挙で数えられる
- 場合の数を確率につなげられる

**前提**：`01_Python基礎`　/　**所要**：約25分　/　**レベル**：統計検定4級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ

## 1. 時系列データ

時間とともに変化するデータ。3つの見方を学びます。
ここでは、ある店の月別売上（万円）を例にします。

In [ ]:
sales = pd.Series([100, 110, 121, 115, 130, 143],  # 月別売上（万円）
    index=['1月','2月','3月','4月','5月','6月'], name='売上')
sales                                          # 中身を表示

### ① 増減率（前の月から何％増えたか）
$$ 増減率 = \frac{今月 - 先月}{先月} \times 100 $$

In [ ]:
growth = sales.pct_change() * 100              # 前の月からの変化率(%)
print(growth.round(1))
print('→ 2月は前月比 +10%、4月はマイナス')

### ② 指数（基準を100としたときの大きさ）
1月を基準(100)にして、各月が基準の何倍かを見ます。単位の違うものを比べやすい。

In [ ]:
index = sales / sales['1月'] * 100             # 1月を100としたときの指数
print(index.round(1))
print('→ 6月は1月の143（=43%増）')

### ③ 移動平均（でこぼこをならす）
数か月の平均をずらしながら計算。短期の上下を消し、大きな流れ（トレンド）を見ます。

In [ ]:
ma3 = sales.rolling(window=3).mean()           # 3か月ごとの移動平均（でこぼこをならす）
print(ma3.round(1))

sales.plot(marker='o', label='売上')           # 元の売上
ma3.plot(marker='s', label='3か月移動平均')     # 移動平均
plt.legend(); plt.title('売上と移動平均'); plt.show()

> 移動平均は、気温・株価・感染者数など「上下が激しいデータの傾向」を見るのに使われます。

### 本物のデータで指数と増減率を読む（e-Stat 消費者物価指数）

ここまでは架空データでした。最後に**政府統計の本物の数字**で指数を体感します。総務省「消費者物価指数（2020年基準）」の全国・総合の年平均を、e-Stat APIから取得します（一度取得すると `data/` に保存され、次回からは即読み込み）。

In [ ]:
import os, pandas as pd

def load_cpi():
    """消費者物価指数（全国・総合, 2020年平均=100）の年平均を返す。
    ① data/ のキャッシュ → ② e-Stat API（要 ESTAT_APP_ID）→ ③ 同梱の実値（抜粋）の順。"""
    path = '../data/estat_cpi_overall.csv'
    if os.path.exists(path):                      # ① 取得済みキャッシュを読む
        return pd.read_csv(path)
    appid = os.environ.get('ESTAT_APP_ID')        # ② キーがあれば e-Stat から取得
    if appid:
        import requests
        r = requests.get('https://api.e-stat.go.jp/rest/3.0/app/json/getStatsData',
            params={'appId': appid, 'statsDataId': '0003427113',
                    'cdTab': '1', 'cdCat01': '0001', 'cdArea': '00000'}, timeout=60)
        vals = r.json()['GET_STATS_DATA']['STATISTICAL_DATA']['DATA_INF']['VALUE']
        rows = [(int(v['@time'][:4]), float(v['$'])) for v in vals if v['@time'].endswith('000000')]
        cpi = pd.DataFrame(rows, columns=['年','CPI']).sort_values('年').reset_index(drop=True)
        os.makedirs('../data', exist_ok=True)
        cpi.to_csv(path, index=False, encoding='utf-8-sig')   # 次回用にキャッシュ
        return cpi
    # ③ ネットもキーも無い環境（Colab等）向けの実値抜粋（総務省 消費者物価指数より）
    yrs  = [1970,1975,1980,1985,1990,1995,2000,2005,2010,2015,2018,2019,2020,2021,2022,2023,2024,2025]
    vals = [30.9,53.1,73.2,83.8,89.6,95.9,97.3,95.2,94.8,98.2,99.5,100.0,100.0,99.8,102.3,105.6,108.5,111.9]
    print('※ ネット/キーが無いため抜粋データで動かします（一部の年のみ）')
    return pd.DataFrame({'年': yrs, 'CPI': vals})

cpi = load_cpi()
print(f'{cpi["年"].min()}〜{cpi["年"].max()}年 / {len(cpi)}件')
cpi.tail()

**指数の意味**：消費者物価指数(CPI)は「2020年の物価＝100」とおいたときの各年の物価水準です。110なら2020年より10%高い、という読み方をします。

In [ ]:
# 基準年を変えても「相対的な動き」は同じ。1990年=100 に置き換えてみる
base1990 = cpi['CPI'] / cpi.loc[cpi['年']==1990, 'CPI'].values[0] * 100
cpi2 = cpi.assign(指数_1990基準=base1990.round(1))
# 前年比（インフレ率, %）＝ CPIが前年から何%動いたか
cpi2['前年比%'] = (cpi['CPI'].pct_change() * 100).round(2)
cpi2.tail(8)

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(cpi['年'], cpi['CPI'], marker='.'); ax1.axhline(100, color='C1', ls='--')
ax1.set_title('消費者物価指数（2020=100）'); ax1.set_xlabel('年'); ax1.set_ylabel('指数')
ax2.bar(cpi['年'], cpi['CPI'].pct_change()*100); ax2.axhline(0, color='gray', lw=.8)
ax2.set_title('前年比（インフレ率, %）'); ax2.set_xlabel('年'); ax2.set_ylabel('%')
plt.tight_layout(); plt.show()

> **指数は「基準年」しだいで数字の見え方が変わります**（2020基準と1990基準では同じ年でも値が違う）。ただし**動きの形（増減）は同じ**です。比べるときは基準年をそろえましょう。
>
> ニュースの「物価が前年より◯%上昇」は、まさにこの**CPIの前年比**のことです。

## 2. 場合の数と樹形図

「何通りあるか」を数えるのが場合の数。枝分かれの図＝**樹形図**で整理します。

例：コインを2回投げると？
```
1回目   2回目   結果
        表 ── 表表
  表 <
        裏 ── 表裏
        表 ── 裏表
  裏 <
        裏 ── 裏裏
```
→ 全部で 2×2 = **4通り**。

In [ ]:
import itertools                               # 組み合わせ・順列の便利ツール
# コイン2回の全パターンを列挙
outcomes = list(itertools.product(['表','裏'], repeat=2))  # 表/裏 を2回ぶん全組合せ
for o in outcomes:
    print(o)                                   # 1パターンずつ表示
print('全', len(outcomes), '通り')             # 総数

### 場合の数から確率へ
「2回とも表」になる確率は、1通り ÷ 4通り = 1/4。

In [ ]:
rng = np.random.default_rng(0)                 # 乱数生成器（seedで固定）
trials = rng.integers(0, 2, size=(10000, 2))   # 0=表,1=裏 を 2回×10000セット
both_head = np.sum(np.all(trials == 0, axis=1))  # 2回とも0(表)だったセット数
print('実験での割合:', both_head / 10000, ' (理論 0.25)')

> **よくある間違い**：「％」と「％ポイント」は別物（20%→24%は **+4%ポイント**、“+20%増”ではない）。また移動平均は両端が計算できず**欠ける**ことに注意。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: 売上が 1月200→2月180 のとき、2月の前月比(%)を ans に
ans = None   # (180-200)/200*100
_check('Q1 前月比%', ans, (180-200)/200*100)

In [ ]:
# Q2: 1月を100としたとき、3月(売上210)の指数を ans に（1月=200）
ans = None   # 210/200*100
_check('Q2 指数', ans, 210/200*100)

In [ ]:
# Q3: コインを3枚投げたときの場合の数（表裏の組合せ）を ans に
ans = None   # 2の3乗
_check('Q3 場合の数', ans, 2**3)

---
## 練習問題

**問1.** 売上 `[200,180,210,240,230,260]`（1〜6月）について、
前月比（増減率）と、1月を100とした指数を計算しよう。

**問2.** `weather.csv` の `気温` の **5日移動平均**を計算し、元データと一緒に折れ線で描こう。

**問3.** サイコロを2回ふるとき、出る目の組み合わせは全部で何通り？
また「合計が7になる」のは何通りで、確率はいくつ？（`itertools.product`を使おう）

**問4.** `load_cpi()` で得たCPIを使い、**2010年=100** に基準を置き換えた指数列を作ろう。さらに直近5年の**前年比（%）**を計算し、物価が最も上がった年を答えよう。

In [ ]:
# 問1


In [ ]:
# 問2
w = pd.read_csv('../data/weather.csv')         # 天気データを読み込む

### 使うデータ：`weather.csv`（2026年6月・30日間の天気）
1行＝1日。気温(℃)と降水量(mm)。

| 日付 | 気温 | 降水量 |
|---|---|---|
| 2026-06-01 | 19.2 | 5.9 |
| 2026-06-02 | 23.2 | 0.0 |
| 2026-06-03 | 22.2 | 4.5 |

In [ ]:
# 問3


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[06_時系列と場合の数 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/06_%E6%99%82%E7%B3%BB%E5%88%97%E3%81%A8%E5%A0%B4%E5%90%88%E3%81%AE%E6%95%B0.md)**

これで**4級の出題範囲をほぼ全てカバー**しました。

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 増減率 | (今−前)÷前 ×100(%) |
| 指数 | 基準を100としたときの大きさ |
| 移動平均 | 数期間の平均をずらして計算 |
| 場合の数 | 何通りあるか |
| 樹形図 | 枝分かれで場合を整理する図 |

In [ ]:
# チートシート（時系列・場合の数）
sales.pct_change()*100         # 増減率(%)
sales / sales.iloc[0] * 100    # 指数（最初=100）
sales.rolling(3).mean()        # 3期間の移動平均
import itertools
list(itertools.product([1,2,3], repeat=2))   # 場合の列挙

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "02_統計_4級/06_時系列と場合の数"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")